In [1]:
import dataclasses
import json
import sys
import time
from pathlib import Path
import pandas as pd
from kafka import KafkaProducer

In [2]:
df = pd.read_parquet('green_tripdata_2025-10.parquet')

In [3]:
columns = ['lpep_pickup_datetime',
'lpep_dropoff_datetime',
'PULocationID',
'DOLocationID',
'passenger_count',
'trip_distance',
'tip_amount',
'total_amount']

df = df[columns]

df['lpep_pickup_datetime'] = df['lpep_pickup_datetime'].astype(str)
df['lpep_dropoff_datetime'] = df['lpep_dropoff_datetime'].astype(str)


In [4]:
df.head(10)

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12
5,2025-09-30 21:49:46,2025-10-01 21:18:42,129,37,1.0,7.13,9.42,47.12
6,2025-10-01 00:11:24,2025-10-01 00:18:48,95,134,1.0,1.13,2.08,13.88
7,2025-10-01 00:07:08,2025-10-01 00:21:46,95,70,1.0,6.01,5.72,34.32
8,2025-10-01 00:36:08,2025-10-01 00:54:59,181,137,1.0,6.45,6.98,41.88
9,2025-10-01 00:26:55,2025-10-01 00:33:00,74,236,1.0,1.74,2.91,17.46


In [5]:
df.dtypes

lpep_pickup_datetime      object
lpep_dropoff_datetime     object
PULocationID               int32
DOLocationID               int32
passenger_count          float64
trip_distance            float64
tip_amount               float64
total_amount             float64
dtype: object

In [6]:

def ride_serializer(row):
    return row.to_json().encode('utf-8')

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)
t0 = time.time()

topic_name = 'green-trips'

for _, row in df.iterrows():
    producer.send(topic_name, value=row)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

took 8.49 seconds


In [7]:
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'green-trips',
    bootstrap_servers=['localhost:9092'],
    auto_offset_reset='earliest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    consumer_timeout_ms=1000
)

count = 0
for msg in consumer:
    if msg.value.get('trip_distance', 0) > 5.0:
        count += 1

print(f"Total rides > 5km: {count}")

Total rides > 5km: 8506
